# Aula 14 - Revisão e Futuro

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**06/10/2026 - Sprint 5 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula14.ipynb)

## O que este notebook é

A Aula 12 deixou pronto o `Pipeline` exportado em `app/modelo_aula12.joblib`: regressão linear
sobre o alvo em nível, onze features da base mensal, reajustada sobre 339 meses, com **2,86%** de
MAPE medido no objeto avaliado. Hoje esse arquivo ganha uma tela, que é o app Streamlit de
`app/app.py`.

Este notebook roda **a mesma lógica do app, sem Streamlit**. Toda conta que a tela mostra sai de
`app/logica.py`, e o notebook importa esse módulo. Isso tem duas vantagens: funciona no Colab, onde
subir um servidor Streamlit é incômodo, e deixa cada número do app conferível célula a célula.

São cinco partes:

1. carregar o modelo e a base;
2. histórico contra previsão nos 24 meses de teste, e qual número de erro acompanha a tela;
3. a previsão do próximo mês, a única genuinamente futura;
4. cenários, e o que acontece quando o cenário mexe numa entrada que o modelo não lê;
5. como rodar o app de verdade, e o exercício da linha do tempo do case.

## 1. O modelo e a base

A célula abaixo encontra `app/logica.py`, o modelo e os CSVs. No repositório clonado, eles estão
em `../app/` e `../dados/mensal/`. No Colab, a célula baixa os mesmos arquivos do GitHub para a
pasta de trabalho, com a mesma estrutura de pastas, e o módulo passa a achá-los sozinho.

In [ ]:
import os
import sys
import urllib.request
import warnings

BRUTA = "https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/main/"
SERIES = ["abate_bovinos", "abate_suinos", "abate_frangos", "producao_ovos", "producao_leite"]

if os.path.exists(os.path.join("..", "app", "logica.py")):
    PASTA_APP = os.path.join("..", "app")
else:
    # Colab: recria app/ e dados/mensal/ na pasta de trabalho
    arquivos = (["app/logica.py", "app/modelo_aula12.joblib"]
                + ["dados/mensal/%s.csv" % s for s in SERIES])
    for relativo in arquivos:
        if not os.path.exists(relativo):
            os.makedirs(os.path.dirname(relativo), exist_ok=True)
            try:
                urllib.request.urlretrieve(BRUTA + relativo, relativo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). Se a rede da sala "
                    "falhou, peca as pastas 'app' e 'dados' para uma dupla que tenha o "
                    "repositorio clonado e coloque as duas ao lado deste notebook. Depois, "
                    "rode esta celula de novo." % (relativo, erro)
                ) from erro
    PASTA_APP = "app"

sys.path.insert(0, PASTA_APP)
import logica

base = logica.base_analitica()
pipe = logica.carregar_pipeline()

print("base   : %d linhas, de %s a %s" % (len(base), base["periodo"].iloc[0],
                                         base["periodo"].iloc[-1]))
print("modelo :", pipe)
print("features:", logica.FEATURES)

Se aparecer um `InconsistentVersionWarning` na carga, o `scikit-learn` da máquina é de outra
versão que a que gravou o arquivo (1.9.1). A Aula 12 mediu esse caso: três avisos, previsão idêntica
bit a bit. O aviso é a única sinalização da divergência, então não silencie.

## 2. Histórico contra previsão, e o número que acompanha a tela

`prever_teste` devolve os 24 meses de teste (2024-04 a 2026-03) com o valor medido pelo IBGE e a
previsão do modelo exportado.

In [ ]:
teste = logica.prever_teste(pipe, base)
print(teste.assign(real=(teste["real"] / 1e6).round(1),
                   previsto=(teste["previsto"] / 1e6).round(1),
                   erro_pct=teste["erro_pct"].round(2)).to_string(index=False))

Agora a pergunta que decide o que vai escrito na tela: qual MAPE acompanha este gráfico?

In [ ]:
mape_exportado_no_teste = logica.mape(teste["real"], teste["previsto"])
mape_honesto = logica.mape_do_avaliado(base)

print("MAPE do objeto exportado nos 24 meses : %.2f%%" % mape_exportado_no_teste)
print("MAPE do objeto avaliado nos 24 meses  : %.2f%%" % mape_honesto)

O exportado erra **2,65%** nos 24 meses porque treinou com eles: foi reajustado sobre as 339 linhas
depois de a avaliação terminar. Esse número não estima desempenho futuro nenhum. O que vai para a
tela é o **2,86%**, medido no objeto avaliado, que treinou em 315 meses e nunca viu os 24 de teste.
É a mesma distinção da Aula 12, e o app a mantém: o cartão de MAPE mostra o avaliado, e a legenda
do gráfico diz que o erro mês a mês é do exportado.

Um segundo cuidado: **as 24 previsões são de um passo à frente.** Cada mês usa o `lag1` real, o
abate medido no mês anterior. Não é a previsão de 24 meses de uma vez que o TAPI da LDC pede; a Aula
05 mediu que prever o horizonte inteiro de forma recursiva erra bem mais.

O gráfico abaixo é o mesmo do app, feito com matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

tabela = logica.tabela_do_grafico(pipe, base, 48, "patamar_frango", 0)
fig, ax = plt.subplots(figsize=(11, 4.5))
for curva, cor in [("Medido pelo IBGE", "#2e2640"), ("Previsão do modelo", "#ff4545")]:
    parte = tabela[tabela["curva"] == curva]
    ax.plot(parte["data"], parte["valor"], color=cor, linewidth=2, label=curva)
ax.set_ylabel("milhões de kg por mês")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. A única previsão genuinamente futura

O modelo precisa, para prever um mês, do abate de frango de 1, 2, 3 e 12 meses antes e das outras
quatro séries no mês anterior. Para o mês seguinte ao último do CSV, todas essas entradas **já foram
medidas**. Para dois meses adiante, `lag1` seria a própria previsão, e as outras séries precisariam
de modelo próprio.

In [ ]:
periodo, X = logica.features_do_proximo_mes()
for nome, valor in zip(logica.FEATURES, X[0]):
    print("%-20s %18.2f" % (nome, valor))

periodo, valor = logica.prever_proximo_mes(pipe)
print()
print("previsão para %s: %s kg" % (periodo, logica.numero_br(valor, 0)))

## 4. Cenários

Um cenário, no app, é um choque percentual aplicado a entradas que o modelo lê, nos 24 meses de
teste, com a previsão recalculada. É análise de sensibilidade do modelo: responde "quanto a
previsão muda se esta entrada tivesse sido X% diferente". Ela não prevê o que aconteceria na
economia.

### 4.1 O cenário que um gestor pediria primeiro, e que o modelo não aceita

A pergunta natural na LDC é "e se o milho subir 10%?". Rode a célula abaixo.

In [ ]:
X_teste = base[logica.FEATURES].to_numpy()[-logica.N_TESTE:]
try:
    logica.aplicar_choque(X_teste, ["preco_milho"], 10)
except ValueError as erro:
    print("ValueError:", erro)

O erro é de propósito. O modelo tem onze entradas, e nenhuma é preço. Um controle de "preço do
milho" na tela mexeria num número que o `Pipeline` nunca lê, e a previsão não mudaria um
quilograma: seria um controle decorativo, que ensina ao usuário uma relação que o modelo não tem.
`aplicar_choque` recusa em voz alta em vez de ignorar em silêncio. O que faltaria para o milho
entrar está listado no próprio módulo:

In [ ]:
for nome, motivo in logica.ENTRADAS_FORA_DO_MODELO.items():
    print("- %s: %s" % (nome, motivo))

### 4.2 O mesmo cenário em dois ajustes do mesmo modelo

A Aula 12 mediu que reajustar o modelo de 315 para 339 meses desloca os coeficientes numa mediana de
17,2% e faz o de `abate_bovinos_lag1` trocar de sinal, enquanto a previsão se move menos de 1%. Um
cenário é uma leitura de coeficiente. A célula abaixo aplica o mesmo choque de +10% nos dois
ajustes e compara.

In [ ]:
cenarios = {
    "patamar do frango (lag1, lag2, lag3, lag12)": "patamar_frango",
    "outras 4 séries no mês anterior": "outras_proteinas",
    "só lag1": ["lag1"],
    "só bovinos no mês anterior": "so_bovinos",
}
linhas = []
for rotulo, cenario in cenarios.items():
    e = logica.estabilidade_do_cenario(pipe, base, cenario, 10)
    linhas.append({"cenário": rotulo,
                   "exportado (%)": round(e["exportado"], 3),
                   "avaliado (%)": round(e["avaliado"], 3),
                   "mesmo sinal": e["mesmo_sinal"]})
print(pd.DataFrame(linhas).to_string(index=False))

Leia a tabela de cima para baixo:

- **O choque no patamar do frango** move a previsão em cerca de 9,6% nos dois ajustes, com
  diferença na terceira casa. As quatro defasagens são colineares (VIF de até 69, na Aula 12), e
  o modelo reparte peso entre elas de um jeito que muda a cada reajuste; somadas, dão quase o
  mesmo número.
- **Só `lag1`** é a mesma família de entradas lida isoladamente, e o efeito quase dobra de um
  ajuste para o outro.
- **Só bovinos** troca de sinal. É o coeficiente que a Aula 12 viu inverter, agora visto como
  cenário: com um ajuste, "mais bovinos no mês anterior aumenta o frango previsto"; com o outro,
  diminui.

O critério que sai daqui para a ART.9 é o mesmo que a Aula 12 deixou para a ART.8, aplicado à tela:
**um cenário só vai para a apresentação se o efeito tiver o mesmo sinal e a mesma ordem de grandeza
nos dois ajustes.** O app faz essa conta a cada movimento do controle, e mostra um aviso quando o
sinal troca.

O modelo é linear, então o efeito é proporcional ao choque. Confira:

In [ ]:
for choque in (-20, -10, 10, 20):
    print("choque de %+3d%% no patamar: efeito médio de %+.3f%%"
          % (choque, logica.efeito_medio(pipe, base, "patamar_frango", choque)))

## 5. Rodar o app de verdade

O app vive em `app/app.py` e só decide o que aparece na tela; as contas são as deste notebook. Para
rodar, **na sua máquina e a partir da raiz do repositório**:

```bash
python3 -m pip install -r app/requirements.txt
streamlit run app/app.py
```

O navegador abre em `http://localhost:8501`. A barra lateral tem a escolha da série, a janela do
histórico, a entrada do cenário e o choque. Escolhendo uma série que não é frango, o app mostra só o
histórico, com um aviso: o modelo prevê só o abate de frangos, e as outras séries entram nele como
entrada defasada.

A estrutura mínima de um app Streamlit, que é o ponto de partida da prática das 10h45, cabe em
poucas linhas. O Streamlit reexecuta o script inteiro a cada clique:

```python
import streamlit as st
import logica

st.title("Abate de frangos")
serie = st.selectbox("Série", logica.SERIES)
tabela = logica.carregar_series()[serie].set_index("periodo")
st.line_chart(tabela["valor"])
```

No app de `app/`, o `st.line_chart` foi trocado por um gráfico Altair por um motivo visível: o eixo
do `st.line_chart` começa em zero, e um erro de 2,86% em 1,2 bilhão de quilogramas vira uma linha
reta.

### Exercício: a linha do tempo do case

Complete a tabela abaixo com o resultado **da sua dupla** em cada etapa, com o número e o arquivo
onde ele está. Ela é o esqueleto da ART.10.

In [ ]:
linha_do_tempo = pd.DataFrame({
    "etapa": ["Entendimento do negócio (Aulas 01 e 02)", "Exploração (Aula 03)",
              "Base analítica (Aulas 04 e 07)", "Primeiro modelo (Aula 05)",
              "Comparação de modelos (Aulas 07 a 11)", "Pipeline exportado (Aula 12)",
              "App e cenários (Aula 14)"],
    "resultado da dupla": [""] * 7,
    "número que sustenta": [""] * 7,
    "arquivo": [""] * 7,
})
linha_do_tempo